#dataset loading

In [ ]:
!pip install torchcodec

In [ ]:
import pandas as pd

df_female = pd.read_csv("/content/drive/MyDrive/male-female-data_slr143/FemaleVoice.tsv", sep="\t")
df_male = pd.read_csv("/content/drive/MyDrive/male-female-data_slr143/MaleVoice.tsv", sep="\t")


In [ ]:
df_female = df_female.rename(columns={'audio_id': 'path', 'sentence': 'sentence'})
df_male = df_male.rename(columns={'audio_id': 'path', 'sentence': 'sentence'})
df = pd.concat([df_male, df_female], ignore_index=True)
df

,path,sentence
0,Voice64,इष्टमित्रहरूले भनेका थिए- आजकल अङ्ग्रेजी नपढाई...
1,Voice65,मलाई कक्षाका साथीहरू निकै मन पराउथे।
2,Voice66,तिहारको पहिलो दिनलाई काग-तिहार भनिन्छ।
3,Voice67,"छोरोले जब म्याट्रिकमा प्रथम श्रेणी ल्यायो, बाक..."
4,Voice68,"मनिस सम्पत्ति,घरबंगलाका लागि आफ्नो छिमेकी,घरपर..."
...,...,...
670,Voice2011,सबैको घरघरमा नेपाल समाचारपत्र आउथ्यौँ।
671,Voice2013,कलेज सकिएपछी उनिहरु पहिलोपल्ट भेट्न लागिरहेका ...
672,Voice2014,कस्तो होला त्यो भेट भन्ने प्रश्न सबैको मनमस्ति...
673,Voice2020,सबैले उनिहरुको ताली पिटेर स्वागत गरे।


In [ ]:
dataset_path = "/content/drive/MyDrive/male-female-data_slr143/"

df["path"] = dataset_path+df["path"] + ".wav"
df.head(1)

,path,sentence
0,/content/drive/MyDrive/male-female-data_slr143...,इष्टमित्रहरूले भनेका थिए- आजकल अङ्ग्रेजी नपढाई...


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['path', 'sentence'],
    num_rows: 675
})

In [ ]:
from datasets import Audio

dataset = dataset.cast_column("path", Audio(sampling_rate=16000))
dataset[0]

{'path': <datasets.features._torchcodec.AudioDecoder at 0x79a88fe5c4a0>,
 'sentence': 'इष्टमित्रहरूले भनेका थिए- आजकल अङ्ग्रेजी नपढाईकन हुँदैन।'}

#Tokenizer and Feature Extractor

In [ ]:
import pandas as pd
import re
from collections import Counter

# merge your male + female TSVs into df before this step

def extract_chars(text):
    return list(text)

all_text = " ".join(df["sentence"].tolist())
chars = extract_chars(all_text)

char_counts = Counter(chars)


In [ ]:
nepali_chars = sorted([c for c in char_counts.keys() if not c.isspace()])


In [ ]:
vocab = {ch: i for i, ch in enumerate(nepali_chars, start=2)}
vocab["|"] = 1     # word separator
vocab["<pad>"] = 0 # padding token


In [ ]:
import json
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)


In [ ]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    unk_token="<unk>",
    pad_token="<pad>",
    word_delimiter_token="|"
)


In [ ]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=False
)


In [ ]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(
    tokenizer=tokenizer,
    feature_extractor=feature_extractor
)



#Data Preprocessing

In [ ]:
import re

chars_to_ignore_regex = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\\…]'  # remove punctuation

def remove_special_characters(batch):
    # Clean text
    text = batch["sentence"].lower()
    text = re.sub(chars_to_ignore_regex, '', text)
    batch["sentence"] = text
    return batch

def prepare_dataset(batch):
    # Audio loading
    # The 'path' column now contains AudioDecoder objects after cast_column
    audio_object = batch["path"]
    speech = audio_object["array"]
    sr = audio_object["sampling_rate"]

    # Resample if needed (though it should already be 16000 from cast_column)
    if sr != 16000:
        speech_tensor = torch.from_numpy(speech).float()
        speech_tensor = torchaudio.functional.resample(speech_tensor, sr, 16000)
        speech = speech_tensor.numpy()

    batch["input_values"] = processor(
        speech, sampling_rate=16000
    ).input_values[0]

    # Encode text
    batch["labels"] = processor.tokenizer(
        batch["sentence"]
    ).input_ids

    return batch

In [ ]:
dataset = dataset.map(remove_special_characters)


Map:   0%|          | 0/675 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names,
    num_proc=4
)


Map (num_proc=4):   0%|          | 0/675 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset.train_test_split(test_size=0.1,seed=42)

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: any
    padding: bool = True

    def __call__(self, batch):
        # input_values
        speech_inputs = [{"input_values": b["input_values"]} for b in batch]
        batch_inputs = self.processor.feature_extractor.pad(
            speech_inputs,
            padding=self.padding,
            return_tensors="pt"
        )

        # labels
        label_features = [{"input_ids": b["labels"]} for b in batch]
        batch_labels = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Replace padding with -100 for CTC
        batch_labels["input_ids"][batch_labels["input_ids"] == self.processor.tokenizer.pad_token_id] = -100

        batch_inputs["labels"] = batch_labels["input_ids"]

        return batch_inputs


In [ ]:
from huggingface_hub import login

login()  # It will ask for your HF token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


#Model Loading

In [ ]:
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, TrainingArguments, Trainer


In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
)

# Adjust to your Nepali vocab
model.lm_head = torch.nn.Linear(model.lm_head.in_features, len(processor.tokenizer), bias=True)
model.lm_head.requires_grad = True


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)


In [ ]:
model.wav2vec2.feature_extractor._freeze_parameters()


In [ ]:
!pip install evaluate jiwer
import evaluate
import numpy as np # Added numpy import

# Load the WER and CER metrics
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    # The Trainer passes an EvalPrediction object
    predictions = pred.predictions # These are raw logits
    labels = pred.label_ids

    # Convert logits to predicted token IDs
    predicted_ids = np.argmax(predictions, axis=-1)

    # Process labels to replace -100 padding with pad_token_id (0) for decoding
    processed_labels = []
    for label_seq in labels:
        label_seq_np = np.array(label_seq) # Convert to numpy array for boolean indexing
        label_seq_np[label_seq_np == -100] = processor.tokenizer.pad_token_id
        processed_labels.append(label_seq_np.tolist()) # Convert back to list

    # Decode predictions and labels
    pred_str = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(processed_labels, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.4 MB/s eta 0:00:00


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./wav2vec2-ne-asr",
    group_by_length=True,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    num_train_epochs=40,
    fp16=True,
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    learning_rate=3e-4,          # IMPORTANT
    warmup_steps=500,
    save_total_limit=4,
    load_best_model_at_end=True, # Load the best model at the end of training
    metric_for_best_model="wer", # Specify WER as the metric to monitor
    greater_is_better=False      # For WER, a lower value is better
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=processor,
    compute_metrics=compute_metrics
)

In [ ]:
model.config.vocab_size = len(processor.tokenizer)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 84, 'bos_token_id': 83}.


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Wer,Cer
500,6.687001,3.190288,0.971330,0.987874
1000,1.339406,0.568410,0.587156,0.213980
1500,0.709777,0.443461,0.481651,0.107347
2000,0.465978,0.502664,0.433486,0.095043
2500,0.327564,0.513006,0.407110,0.089515
3000,0.260375,0.523719,0.392202,0.088088


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3040, training_loss=2.511169475003293, metrics={'train_runtime': 3705.1393, 'train_samples_per_second': 6.553, 'train_steps_per_second': 0.82, 'total_flos': 4.966732581437103e+18, 'train_loss': 2.511169475003293, 'epoch': 40.0})

In [ ]:
# Define folder inside your Drive
save_path = "/content/drive/MyDrive/nepali-asr-xlsr300m/4rd-model_checkpoint"

# Save model
trainer.save_model(save_path)

# Save processor (feature extractor + tokenizer)
processor.save_pretrained(save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/drive/MyDrive/nepali-asr-xlsr300m/4rd-model_checkpoint/processor_config.json']